In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')
sys.path.append('../Stuff')

from Constants import db

cursor = db.cursor()

In [ ]:
from Stuff.DataPrep.DataPrep import DataPrep, PitchType
from Stuff.DataPrep.PrepMap import standard_prep_map

dataPrep = DataPrep(standard_prep_map, PitchType.Changeup)

In [ ]:
from DBTypes import *

pitches = DB_PitchStatcast.Select_From_DB(
    cursor=cursor,
    conditional=dataPrep.conditional_statement + "AND LevelId=1",
    values=()
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from Buckets import *

run_buckets = []
for pitch in pitches:
    value, isinplay = DataPrep.GetInPlayBucket(pitch)
    if isinplay == 1:
        run_buckets.append(value)

In [ ]:
labels = [f"(-∞, {BUCKET_INPLAY_VALUE[0]:.3f}]"]
for i in range(BUCKET_INPLAY_VALUE.size(0)-1):
    left = f"{BUCKET_INPLAY_VALUE[i]:.3f}"
    right = f"{BUCKET_INPLAY_VALUE[i+1]:.3f}"

    if left == "-0.000" and right == "0.000":
        label = "Exactly 0"
    else:
        label = f"({left}, {right}]"
    labels.append(label)
    
labels.append(f"({BUCKET_INPLAY_VALUE[-1].item():.3f}, +∞]")

In [ ]:
num_buckets = BUCKET_INPLAY_VALUE.size(0) - 1
counts = torch.bincount(torch.tensor(run_buckets), minlength=num_buckets).numpy()

plt.figure(figsize=(16, 7))

x_range = range(len(labels))

bars = plt.bar(x_range, counts,
               color='skyblue',
               edgecolor='navy',
               linewidth=1.2,
               alpha=0.9)

plt.title('Histogram — Manually Defined Buckets', fontsize=15, pad=15)
plt.xlabel('Value Buckets', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(x_range, labels, rotation=45, ha='right', fontsize=10)

# Add count on top of each bar
max_count = counts.max()
for bar in bars:
    height = bar.get_height()
    if height > 0:
        plt.text(bar.get_x() + bar.get_width()/2,
                 height + max_count * 0.015,
                 f'{int(height)}',
                 ha='center', va='bottom',
                 fontsize=10,
                 fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_cdf(data: list[float], title="CDF - Both Tails", xlabel="Value"):
    data = np.array(data)
    sorted_data = np.sort(data)
    n = len(sorted_data)
    l = n // 2
    cdf = np.arange(1, n + 1) / n
    
    fig, ax1 = plt.subplots(figsize=(12, 7))
    
    # Left tail: CDF on log scale (blue)
    ax1.plot(sorted_data[:l], cdf[:l], linewidth=2.5, color='blue', 
             marker='.', markersize=4, label='Left Tail (CDF)')
    ax1.set_yscale('log')
    ax1.set_ylabel("Cumulative Probability  (log scale)", color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.set_xlabel(xlabel)
    
    # Right tail: 1 - CDF on log scale (red) using twin axis
    ax2 = ax1.twinx()
    ax2.plot(sorted_data[l:], 1 - cdf[l:], linewidth=2.5, color='red', 
             marker='.', markersize=4, label='Right Tail (1-CDF)')
    ax2.set_yscale('log')
    ax2.set_ylabel("Survival Probability  (1 - CDF, log scale)", color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    
    # Common elementsl 
    fig.suptitle(title, fontsize=14)
    ax1.grid(True, alpha=0.3)
    
    # Add reference lines
    for p in [0.001, 0.01, 0.05, 0.10]:
        ax1.axhline(p, color='blue', linestyle='--', alpha=0.4)
        ax2.axhline(p, color='red', linestyle='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()

In [ ]:
from Stuff.DataPrep.PrepMap import __map_pitch_stuff_fastball, __map_pitch_stuff_curveball, __map_pitch_stuff_changeup
#values = [p.SpinRate for p in pitches]
for i, name in enumerate(["Velocity", "Break Angle", "IVB", "Horiz", "Extension", "X0", "Z0", "SpinRate", "SpinDirection"]):
    values = [__map_pitch_stuff_changeup(p)[i] for p in pitches]
    plot_cdf(values, f"{name} CDF", "units")